In [2]:
# === imports ===
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import BaggingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import tensorflow as tf
from tensorflow.keras import layers, models                                                                                                                                                                                                                                                                                                                                                   
                       

In [3]:
# === Constants / settings ===
DATA_PATH = "PhiUSIIL_Phishing_URL_Dataset.csv"     # <- update to your file
SELECTED_FEATURES = [
    "IsDomainIP","IsHTTPS","URLLength","DomainLength","NoOfSubDomain",
    "ObfuscationRatio","URLSimilarityIndex","URLTitleMatchScore",
    "HasExternalFormSubmit","HasPasswordField","HasHiddenFields",
    "NoOfiFrame","NoOfPopup","NoOfURLRedirect","Bank","Pay","Crypto",
    "TLDLegitimateProb","LetterRatioInURL"
]
RANDOM_STATE = 42
TEST_SIZE = 0.2




**FishNet Feature Explanations and Implementation**

* **IsDomainIP:** Checks if the site uses an IP address instead of a name — phishing sites often do this.
  *Implementation:* Use regex on the URL to detect numeric IP patterns.

* **IsHTTPS:** Secure sites use HTTPS; missing it can mean lower trust.
  *Implementation:* Read `window.location.protocol` and flag if not “https:”.

* **URLLength:** Long URLs often hide fake links or tracking strings.
  *Implementation:* Use `url.length` to measure total URL characters.

* **DomainLength:** Very long domains can be suspicious or cloned.
  *Implementation:* Extract the domain and count its characters.

* **NoOfSubDomain:** Too many subdomains are common in fake login pages.
  *Implementation:* Split domain by “.” and count all parts beyond the main domain.

* **ObfuscationRatio:** Measures how many weird or encoded characters are in the URL.
  *Implementation:* Scan URL for special symbols, punycode, or Unicode chars and compute ratio.

* **URLSimilarityIndex:** Compares the link to known real sites — low similarity suggests a fake.
  *Implementation:* Compare domain text with a local or cloud list of verified domains.

* **URLTitleMatchScore:** Checks if the page title matches the domain — mismatches look fishy.
  *Implementation:* Compare `document.title` and domain name using string similarity.

* **HasExternalFormSubmit:** Detects if a form sends data to another domain — a red flag.
  *Implementation:* Inspect all `<form>` tags; check if `action` domain differs from current site.

* **HasPasswordField:** Finds password boxes that may collect credentials.
  *Implementation:* Detect `<input type="password">` elements in the DOM.

* **HasHiddenFields:** Hidden form inputs can be used to steal or track info secretly.
  *Implementation:* Detect `<input type="hidden">` elements.

* **NoOfiFrame:** Too many iframes can hide fake content inside pages.
  *Implementation:* Count all `<iframe>` elements in the document.

* **NoOfPopup:** Many pop-ups are used to trick users into entering data.
  *Implementation:* Track calls to `window.open()` or modal dialogs via event listeners.

* **NoOfURLRedirect:** Multiple redirects can mask the real site.
  *Implementation:* Use `chrome.webRequest` API to log redirect counts before page load.

* **Bank / Pay / Crypto:** Flags financial keywords often used in scams.
  *Implementation:* Search the visible text and HTML for these words.

* **TLDLegitimateProb:** Some domain endings (.xyz, .top) are riskier than normal ones (.com).
  *Implementation:* Extract TLD and compare with a local “trusted vs. risky” lookup table.

* **LetterRatioInURL:** Too many numbers or symbols instead of letters usually means the site isn’t real.
  *Implementation:* Compute ratio of alphabetic to total characters in the URL.

---

All these features are lightweight enough for the extension to extract instantly.
They form a **compact feature vector** that is then sent to the **FishNet cloud ML model** for real-time phishing detection.


In [4]:
# === 1. Load & EDA ===
df = pd.read_csv(DATA_PATH)
print("Initial shape:", df.shape)
print(df.isnull().sum())

# Drop rows with missing data in selected features or label
df = df.dropna(subset=SELECTED_FEATURES + ["label"])
print("After dropping missing:", df.shape)

# (Optional) Inspect class balance
print(df['label'].value_counts())


Initial shape: (235795, 56)
FILENAME                      0
URL                           0
URLLength                     0
Domain                        0
DomainLength                  0
IsDomainIP                    0
TLD                           0
URLSimilarityIndex            0
CharContinuationRate          0
TLDLegitimateProb             0
URLCharProb                   0
TLDLength                     0
NoOfSubDomain                 0
HasObfuscation                0
NoOfObfuscatedChar            0
ObfuscationRatio              0
NoOfLettersInURL              0
LetterRatioInURL              0
NoOfDegitsInURL               0
DegitRatioInURL               0
NoOfEqualsInURL               0
NoOfQMarkInURL                0
NoOfAmpersandInURL            0
NoOfOtherSpecialCharsInURL    0
SpacialCharRatioInURL         0
IsHTTPS                       0
LineOfCode                    0
LargestLineLength             0
HasTitle                      0
Title                         0
DomainTitleM

In [5]:

# === 2. Feature / target preparation ===
X = df[SELECTED_FEATURES]
y = df['label']

# Encode any categorical features if needed (example: TLD may need encoding)
# For simplicity assume features are numeric; if not, apply one-hot or label encoding here.

# Split train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

# Scale features for models like SVM & ANN
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [6]:

# === 3. Model training & evaluation helpers ===
def eval_model(name, model, X_t, y_t, X_s, y_s):
    model.fit(X_t, y_t)
    y_pred = model.predict(X_s)
    print(f"=== {name} results ===")
    print("Accuracy:", accuracy_score(y_s, y_pred))
    print("Precision:", precision_score(y_s, y_pred))
    print("Recall:", recall_score(y_s, y_pred))
    print("F1:", f1_score(y_s, y_pred))
    try:
        y_prob = model.predict_proba(X_s)[:,1]
        print("AUC:", roc_auc_score(y_s, y_prob))
    except:
        pass
    print()


In [7]:
# === 4. Logistic Regression ===
lr = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000)
eval_model("Logistic Regression", lr, X_train_scaled, y_train, X_test_scaled, y_test)


=== Logistic Regression results ===
Accuracy: 0.9996819270976908
Precision: 0.9994441356309061
Recall: 1.0
F1: 0.9997219905476786
AUC: 0.9999708731354109



In [8]:
# === 5. SVM ===
svm = SVC(probability=True, random_state=RANDOM_STATE)
eval_model("SVM", svm, X_train_scaled, y_train, X_test_scaled, y_test)


=== SVM results ===
Accuracy: 0.9996607222375369
Precision: 0.9995922301304864
Recall: 0.9998146088246199
F1: 0.9997034071108145
AUC: 0.9998987754448675



In [10]:
# === 6. Bagging (with base estimator = logistic) ===
bag = BaggingClassifier(estimator=LogisticRegression(max_iter=1000),
                        n_estimators=50, random_state=RANDOM_STATE)
eval_model("Bagging (Logistic base)", bag, X_train_scaled, y_train, X_test_scaled, y_test)


=== Bagging (Logistic base) results ===
Accuracy: 0.9996819270976908
Precision: 0.9994441356309061
Recall: 1.0
F1: 0.9997219905476786
AUC: 0.999973335957405



In [12]:
# === 7. XGBoost ===
xgb = XGBClassifier(tree_method='hist',
                    use_label_encoder=False, eval_metric='logloss',
                    random_state=RANDOM_STATE)
eval_model("XGBoost", xgb, X_train, y_train, X_test, y_test)


c:\Users\Hi\Documents\GitHub\FishNet\env\Lib\site-packages\xgboost\training.py:199: UserWarning: [20:36:09] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


=== XGBoost results ===
Accuracy: 0.9998091562586144
Precision: 0.9996664072056044
Recall: 1.0
F1: 0.9998331757771228
AUC: 0.999985959527111



In [13]:
# === 8. ANN (Keras) ===
model_ann = models.Sequential([
    layers.Input(shape=(X_train_scaled.shape[1],)),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(1, activation='sigmoid')
])
model_ann.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_ann.fit(X_train_scaled, y_train, epochs=20, batch_size=256, validation_split=0.1)
y_pred_ann = (model_ann.predict(X_test_scaled) > 0.5).astype(int)
print("=== ANN results ===")
print("Accuracy:", accuracy_score(y_test, y_pred_ann))
print("Precision:", precision_score(y_test, y_pred_ann))
print("Recall:", recall_score(y_test, y_pred_ann))
print("F1:", f1_score(y_test, y_pred_ann))

# === 9. Save best model (placeholder) ===
# e.g., joblib.dump(best_model, 'fishnet_model.pkl')


Epoch 1/20
664/664 ━━━━━━━━━━━━━━━━━━━━ 1s 921us/step - accuracy: 0.9791 - loss: 0.0571 - val_accuracy: 0.9990 - val_loss: 0.0032
Epoch 2/20
664/664 ━━━━━━━━━━━━━━━━━━━━ 1s 824us/step - accuracy: 0.9993 - loss: 0.0034 - val_accuracy: 0.9994 - val_loss: 0.0022
Epoch 3/20
664/664 ━━━━━━━━━━━━━━━━━━━━ 1s 817us/step - accuracy: 0.9996 - loss: 0.0020 - val_accuracy: 0.9995 - val_loss: 0.0020
Epoch 4/20
664/664 ━━━━━━━━━━━━━━━━━━━━ 1s 835us/step - accuracy: 0.9998 - loss: 0.0014 - val_accuracy: 0.9996 - val_loss: 0.0019
Epoch 5/20
664/664 ━━━━━━━━━━━━━━━━━━━━ 1s 816us/step - accuracy: 0.9998 - loss: 0.0012 - val_accuracy: 0.9996 - val_loss: 0.0016
Epoch 6/20
664/664 ━━━━━━━━━━━━━━━━━━━━ 1s 827us/step - accuracy: 0.9998 - loss: 0.0011 - val_accuracy: 0.9997 - val_loss: 0.0015
Epoch 7/20
664/664 ━━━━━━━━━━━━━━━━━━━━ 1s 825us/step - accuracy: 0.9999 - loss: 0.0011 - val_accuracy: 0.9997 - val_loss: 0.0013
Epoch 8/20
664/664 ━━━━━━━━━━━━━━━━━━━━ 1s 829us/step - accuracy: 0.9999 - loss: 9.2884e-0

In [14]:
import joblib

joblib.dump(model_ann, 'ann.pkl')

['ann.pkl']